# Regular tetrahedron: current public study

Current reader stage: **2026-11-10**. Start here when reading this unit in Jupyter.

Four certified hull patches at 1.2 Å remain distinct from one native link and zero public Mouths. GHECOM/CAVER radii for these inputs are resolved by the later reader audit: 1.87/1.70 Å.

Read [the system](system.ipynb), [the frozen native study](dfnd.ipynb),
then return here for actual public receiving and the [later certificate](boundary_connectivity.md).
The [related follow-up](atomic_models.md) and [case overview](index.md) retain the evidence history.
We stay with these three inputs before the tube. Physical refinement #151, volume #75
and general public support #60 remain open.

Run All from this notebook directory in a fresh kernel. It recomputes DFND through
the public API and replays existing independent witnesses, without overwriting
frozen files or running third parties/servers. Use the complete study bundle,
which includes the parent shell, input files and reference helpers.


In [1]:
import json
import sys
from pathlib import Path
from IPython.display import Markdown, display

case_directory = Path.cwd()
if case_directory.name != 'regular_tetrahedron':
    raise ValueError('Run from the notebook case directory')
sys.path.insert(0, str(case_directory.parent))
import public_study as study
import topomt as tmt

probes = [1.2, 1.4, 1.7]
rows = []
for probe in probes:
    result = study.receive(case_directory, probe)
    rows.append({'probe_angstroms': probe, **study.describe(result)})
display(Markdown('| Probe (Å) | Public types | Native links | Public Mouths | Physical view |\n'
    '|---|---|---|---|---|\n' + '\n'.join(
    f"| {row['probe_angstroms']:g} | {', '.join(row['public_types']) or 'Empty'} | {row['native_exterior_links']} | {row['public_mouths']} | {row['physical_view_status']} |"
    for row in rows)))

| Probe (Å) | Public types | Native links | Public Mouths | Physical view |
|---|---|---|---|---|
| 1.2 | Percolating | 1 | 0 | not_available |
| 1.4 | Void | 0 | 0 | not_available |
| 1.7 | Empty | 0 | 0 | not_available |

## Inspect the public objects

These are actual Python classes, registry relations and contextual supports. Legacy dimensionality is a catalog convention. A lining atom set or carrier support does not establish an exact solvent volume.

In [2]:
topography = study.receive(case_directory, 1.4)
for feature in topography.values():
    display(Markdown(f"**{type(feature).__name__}** — {feature.feature_id}"))
    print('Type/shape/legacy dimension:', feature.feature_type, feature.shape_type, feature.dimensionality)
    print('Parents:', [item.feature_id for item in topography.parents_of(feature.feature_id)])
    print('Children:', [item.feature_id for item in topography.children_of(feature.feature_id)])
    print('Neutral support:', getattr(feature, 'support_key', 'not available; see #60'))
    print('Native contact support:', getattr(feature, 'external_link_support_key', 'not applicable'))
print('Selected source atoms:', topography.input_context.atom_indices.tolist())

# Recover the native carrier geometry; it is not the excluded-atom free volume.
for region in topography.get_features(by='dimensionality', value=2):
    component = topography.dfnd.dfn.components[region.component_id]
    assert region.support_key == component.support_key
    network = topography.dfnd.network
    cell_ids = list(map(int, component.node_indices))
    source_atoms = network.atom_indices_map[network.tetra_atoms[cell_ids]].tolist()
    assert {atom for cell in source_atoms for atom in cell} <= set(topography.input_context.atom_indices)
    vertices = study.puw.get_value(
        study.puw.quantity(network.atom_coords[network.tetra_atoms[cell_ids]], 'nm'),
        to_unit='angstroms')
    print('Transit carrier vertices (Å), array shape:', vertices.shape)
    print('First source-atom tetrahedron:', source_atoms[0])
for mouth in topography.get_features(by='type', value='mouth'):
    print('Native source faces:', mouth.faces)
    print('Carrier triangle area (Å²):', study.puw.get_value(mouth.area, to_unit='angstroms**2'))
    print('Free aperture area is not supplied by this descriptor.')


**Void** — VOI-1

Type/shape/legacy dimension: void concavity 2
Parents: []
Children: []
Neutral support: support-24d62856bfcf1370a5afefd9c5686c620ab4194469cd4b98405aa627030b7d70
Native contact support: not applicable
Selected source atoms: [0, 1, 2, 3]
Transit carrier vertices (Å), array shape: (1, 4, 3)
First source-atom tetrahedron: [0, 1, 2, 3]


## Independently replay the later certificate

The checker reads the original witnesses and verifies the same recorded reference. It neither calls DFND nor accepts the detector label as truth. The certificate is retrospective characterization, not a held-out evaluation. Its floating-point safety margins are distinct from formal interval arithmetic.

In [3]:
verified_reference = study.reference(case_directory)
print(json.dumps(verified_reference, indent=2))

{
  "verification": "passed",
  "opening_component_count": 4,
  "blocked_edge_count": 6,
  "origin_connected_patch_count": 4,
  "origin": {
    "manifest": {
      "format": "qrec/0.3",
      "field": "origin",
      "unit": "angstrom",
      "si": {
        "factor": 1e-10,
        "offset": 0.0,
        "exponents": {
          "m": 1
        }
      },
      "dtype": "float64",
      "shape": [
        3
      ],
      "blocks": [
        3
      ]
    },
    "values": [
      0.0,
      0.0,
      0.0
    ],
    "block_digests": [
      "blake2b-128:afbcb3a85cfb87bffed6f2d5f2709be5"
    ],
    "digest": "blake2b-128:eb18cb501a2158fd31010a99c8d82566"
  },
  "origin_clearance": {
    "manifest": {
      "format": "qrec/0.3",
      "field": "origin_clearance",
      "unit": "angstrom",
      "si": {
        "factor": 1e-10,
        "offset": 0.0,
        "exponents": {
          "m": 1
        }
      },
      "dtype": "float64",
      "shape": [],
      "blocks": [
        1
      ]


## Reporting is a separate choice

Hiding a region with min_size does not close its opening or change its geometry. The large-probe empty controls above instead have no resident wet component.

In [4]:
component = topography.dfnd.raw['wet_components'][0]
cutoff = component['n_nodes'] + 1
filtered = study.receive(case_directory, 1.4, min_size=cutoff)
assert not filtered
assert filtered.dfnd.raw['parameters']['result_key'] == topography.dfnd.raw['parameters']['result_key']
assert filtered.dfnd.raw['wet_components'][0]['support_key'] == component['support_key']
print('The public registry is hidden; the same raw support and physical query remain.')

The public registry is hidden; the same raw support and physical query remain.


## Current conclusion and next work

Four certified hull patches at 1.2 Å remain distinct from one native link and zero public Mouths. GHECOM/CAVER radii for these inputs are resolved by the later reader audit: 1.87/1.70 Å.

The physical view is still unavailable in this native/public implementation;
the independently certified count is a separate observation. Existing carrier
triangle areas are not free aperture areas. Continue physical patch/path work
under [#151](https://github.com/uibcdf/topomt/issues/151), then volume precision
[#75](https://github.com/uibcdf/topomt/issues/75).

[System](system.ipynb) · [Native history](dfnd.ipynb) · [Current certificate](boundary_connectivity.md) · [Overview](index.md).
